In [1]:
import os

In [2]:
os.listdir('.')

['.git',
 '.ipynb_checkpoints',
 'backend.py',
 'data',
 'notebook.ipynb',
 'pages',
 'python-env',
 'style.css',
 'ui',
 'utils']

In [3]:
%matplotlib inline

In [4]:
from utils.imports import pd, np, plt
from data.loaders import get_dataframe
from geopy.distance import geodesic
import seaborn as sns


In [5]:
url = r"https://raw.githubusercontent.com/uber-web/kepler.gl-data/master/nyctrips/data.csv"

df = get_dataframe(url, ",")

In [6]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,fare_amount,tip_amount,total_amount
0,2,2015-01-15 19:05:39 +00:00,2015-01-15 19:23:42 +00:00,1,1.59,-73.993896,40.750111,-73.974785,40.750618,12.0,3.25,17.05
1,2,2015-01-15 19:05:39 +00:00,2015-01-15 19:32:00 +00:00,1,2.38,-73.976425,40.739811,-73.983978,40.757889,16.5,4.38,22.68
2,2,2015-01-15 19:05:40 +00:00,2015-01-15 19:21:00 +00:00,5,2.83,-73.968704,40.754246,-73.955124,40.786858,12.5,0.00,14.30
3,2,2015-01-15 19:05:40 +00:00,2015-01-15 19:28:18 +00:00,5,8.33,-73.863060,40.769581,-73.952713,40.785782,26.0,8.08,41.21
4,2,2015-01-15 19:05:41 +00:00,2015-01-15 19:20:36 +00:00,1,2.37,-73.945541,40.779423,-73.980850,40.786083,11.5,0.00,13.30


In [7]:
df.shape

(97986, 12)

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 97986 entries, 0 to 97985
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   VendorID               97986 non-null  int64  
 1   tpep_pickup_datetime   97986 non-null  str    
 2   tpep_dropoff_datetime  97986 non-null  str    
 3   passenger_count        97986 non-null  int64  
 4   trip_distance          97986 non-null  float64
 5   pickup_longitude       97986 non-null  float64
 6   pickup_latitude        97986 non-null  float64
 7   dropoff_longitude      97986 non-null  float64
 8   dropoff_latitude       97986 non-null  float64
 9   fare_amount            97986 non-null  float64
 10  tip_amount             97986 non-null  float64
 11  total_amount           97986 non-null  float64
dtypes: float64(8), int64(2), str(2)
memory usage: 13.8 MB


In [9]:
df.isna().mean()*100

VendorID                 0.0
tpep_pickup_datetime     0.0
tpep_dropoff_datetime    0.0
passenger_count          0.0
trip_distance            0.0
pickup_longitude         0.0
pickup_latitude          0.0
dropoff_longitude        0.0
dropoff_latitude         0.0
fare_amount              0.0
tip_amount               0.0
total_amount             0.0
dtype: float64

In [10]:
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

In [11]:
df['duree_trajet'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() // 60

In [12]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,fare_amount,tip_amount,total_amount,duree_trajet
0,2,2015-01-15 19:05:39+00:00,2015-01-15 19:23:42+00:00,1,1.59,-73.993896,40.750111,-73.974785,40.750618,12.0,3.25,17.05,18.0
1,2,2015-01-15 19:05:39+00:00,2015-01-15 19:32:00+00:00,1,2.38,-73.976425,40.739811,-73.983978,40.757889,16.5,4.38,22.68,26.0
2,2,2015-01-15 19:05:40+00:00,2015-01-15 19:21:00+00:00,5,2.83,-73.968704,40.754246,-73.955124,40.786858,12.5,0.00,14.30,15.0
3,2,2015-01-15 19:05:40+00:00,2015-01-15 19:28:18+00:00,5,8.33,-73.863060,40.769581,-73.952713,40.785782,26.0,8.08,41.21,22.0
4,2,2015-01-15 19:05:41+00:00,2015-01-15 19:20:36+00:00,1,2.37,-73.945541,40.779423,-73.980850,40.786083,11.5,0.00,13.30,14.0


In [13]:
df["other_bill"] = df["total_amount"] - (df["fare_amount"] + df["tip_amount"])

In [14]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,fare_amount,tip_amount,total_amount,duree_trajet,other_bill
0,2,2015-01-15 19:05:39+00:00,2015-01-15 19:23:42+00:00,1,1.59,-73.993896,40.750111,-73.974785,40.750618,12.0,3.25,17.05,18.0,1.80
1,2,2015-01-15 19:05:39+00:00,2015-01-15 19:32:00+00:00,1,2.38,-73.976425,40.739811,-73.983978,40.757889,16.5,4.38,22.68,26.0,1.80
2,2,2015-01-15 19:05:40+00:00,2015-01-15 19:21:00+00:00,5,2.83,-73.968704,40.754246,-73.955124,40.786858,12.5,0.00,14.30,15.0,1.80
3,2,2015-01-15 19:05:40+00:00,2015-01-15 19:28:18+00:00,5,8.33,-73.863060,40.769581,-73.952713,40.785782,26.0,8.08,41.21,22.0,7.13
4,2,2015-01-15 19:05:41+00:00,2015-01-15 19:20:36+00:00,1,2.37,-73.945541,40.779423,-73.980850,40.786083,11.5,0.00,13.30,14.0,1.80


In [15]:
df["VendorID"].unique()

array([2, 1])

In [16]:
quanti_col = ["VendorID", "passenger_count", "fare_amount", "tip_amount", "total_amount", "duree_trajet", "other_bill"]

In [17]:
df_quanti = df[quanti_col]

In [18]:
df_quanti.groupby(["VendorID"]).mean()

,passenger_count,fare_amount,tip_amount,total_amount,duree_trajet,other_bill
VendorID,,,,,,
1,1.187581,12.218904,1.713269,15.354195,12.747598,1.422022
2,2.063013,12.412492,1.695608,15.597811,14.388402,1.489711


In [19]:
def calculer_distance_row(row):
    point_depart = (row['pickup_latitude'], row['pickup_longitude'])
    point_arrivee = (row['dropoff_latitude'], row['dropoff_longitude'])
    
    return geodesic(point_depart, point_arrivee).km

In [20]:
df['distance_km'] = df.apply(calculer_distance_row, axis=1)

In [21]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,fare_amount,tip_amount,total_amount,duree_trajet,other_bill,distance_km
0,2,2015-01-15 19:05:39+00:00,2015-01-15 19:23:42+00:00,1,1.59,-73.993896,40.750111,-73.974785,40.750618,12.0,3.25,17.05,18.0,1.80,1.614995
1,2,2015-01-15 19:05:39+00:00,2015-01-15 19:32:00+00:00,1,2.38,-73.976425,40.739811,-73.983978,40.757889,16.5,4.38,22.68,26.0,1.80,2.106435
2,2,2015-01-15 19:05:40+00:00,2015-01-15 19:21:00+00:00,5,2.83,-73.968704,40.754246,-73.955124,40.786858,12.5,0.00,14.30,15.0,1.80,3.798685
3,2,2015-01-15 19:05:40+00:00,2015-01-15 19:28:18+00:00,5,8.33,-73.863060,40.769581,-73.952713,40.785782,26.0,8.08,41.21,22.0,7.13,7.779163
4,2,2015-01-15 19:05:41+00:00,2015-01-15 19:20:36+00:00,1,2.37,-73.945541,40.779423,-73.980850,40.786083,11.5,0.00,13.30,14.0,1.80,3.070854


In [22]:
df["distance_moyenne"] = ((df["trip_distance"] * 1.60934) + df["distance_km"]) / 2

In [23]:
df["vitesse_en_kmh"] = (df["distance_moyenne"] / df["duree_trajet"]) * 60

In [24]:
df["vitesse_moy_kmh"] = df["vitesse_en_kmh"].replace([np.inf, -np.inf], np.nan)

In [25]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,fare_amount,tip_amount,total_amount,duree_trajet,other_bill,distance_km,distance_moyenne,vitesse_en_kmh,vitesse_moy_kmh
0,2,2015-01-15 19:05:39+00:00,2015-01-15 19:23:42+00:00,1,1.59,-73.993896,40.750111,-73.974785,40.750618,12.0,3.25,17.05,18.0,1.80,1.614995,2.086923,6.956409,6.956409
1,2,2015-01-15 19:05:39+00:00,2015-01-15 19:32:00+00:00,1,2.38,-73.976425,40.739811,-73.983978,40.757889,16.5,4.38,22.68,26.0,1.80,2.106435,2.968332,6.849998,6.849998
2,2,2015-01-15 19:05:40+00:00,2015-01-15 19:21:00+00:00,5,2.83,-73.968704,40.754246,-73.955124,40.786858,12.5,0.00,14.30,15.0,1.80,3.798685,4.176559,16.706234,16.706234
3,2,2015-01-15 19:05:40+00:00,2015-01-15 19:28:18+00:00,5,8.33,-73.863060,40.769581,-73.952713,40.785782,26.0,8.08,41.21,22.0,7.13,7.779163,10.592483,28.888590,28.888590
4,2,2015-01-15 19:05:41+00:00,2015-01-15 19:20:36+00:00,1,2.37,-73.945541,40.779423,-73.980850,40.786083,11.5,0.00,13.30,14.0,1.80,3.070854,3.442495,14.753549,14.753549


In [26]:
df["hour_recup"] = df["tpep_pickup_datetime"].dt.hour
df["hour_drop"] = df["tpep_dropoff_datetime"].dt.hour

In [27]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,fare_amount,tip_amount,total_amount,duree_trajet,other_bill,distance_km,distance_moyenne,vitesse_en_kmh,vitesse_moy_kmh,hour_recup,hour_drop
0,2,2015-01-15 19:05:39+00:00,2015-01-15 19:23:42+00:00,1,1.59,-73.993896,40.750111,-73.974785,40.750618,12.0,3.25,17.05,18.0,1.80,1.614995,2.086923,6.956409,6.956409,19,19
1,2,2015-01-15 19:05:39+00:00,2015-01-15 19:32:00+00:00,1,2.38,-73.976425,40.739811,-73.983978,40.757889,16.5,4.38,22.68,26.0,1.80,2.106435,2.968332,6.849998,6.849998,19,19
2,2,2015-01-15 19:05:40+00:00,2015-01-15 19:21:00+00:00,5,2.83,-73.968704,40.754246,-73.955124,40.786858,12.5,0.00,14.30,15.0,1.80,3.798685,4.176559,16.706234,16.706234,19,19
3,2,2015-01-15 19:05:40+00:00,2015-01-15 19:28:18+00:00,5,8.33,-73.863060,40.769581,-73.952713,40.785782,26.0,8.08,41.21,22.0,7.13,7.779163,10.592483,28.888590,28.888590,19,19
4,2,2015-01-15 19:05:41+00:00,2015-01-15 19:20:36+00:00,1,2.37,-73.945541,40.779423,-73.980850,40.786083,11.5,0.00,13.30,14.0,1.80,3.070854,3.442495,14.753549,14.753549,19,19


In [28]:
df = df.sort_values(by="tpep_pickup_datetime")

In [29]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,fare_amount,tip_amount,total_amount,duree_trajet,other_bill,distance_km,distance_moyenne,vitesse_en_kmh,vitesse_moy_kmh,hour_recup,hour_drop
88452,2,2015-01-15 00:01:32+00:00,2015-01-15 00:16:06+00:00,6,3.90,-74.004898,40.729919,-73.996223,40.681870,14.5,0.0,15.8,14.0,1.3,5.385980,5.831203,24.990871,24.990871,0,0
88453,2,2015-01-15 00:01:32+00:00,2015-01-15 00:10:35+00:00,2,1.57,-73.970383,40.764954,-73.989159,40.756969,8.5,1.0,10.8,9.0,1.3,1.816494,2.171579,14.477191,14.477191,0,0
88461,2,2015-01-15 00:01:33+00:00,2015-01-15 00:25:32+00:00,6,7.01,-74.001862,40.741180,-73.951241,40.662281,23.5,2.4,27.2,23.0,1.3,9.750319,10.515896,27.432773,27.432773,0,0
88454,2,2015-01-15 00:01:33+00:00,2015-01-15 00:08:25+00:00,1,0.86,-73.985321,40.747009,-73.974289,40.743309,6.5,1.0,8.8,6.0,1.3,1.018338,1.201185,12.011851,12.011851,0,0
88455,2,2015-01-15 00:01:33+00:00,2015-01-15 00:03:21+00:00,1,0.54,-73.981331,40.779167,-73.976700,40.785885,3.5,1.0,5.8,1.0,1.3,0.842213,0.855628,51.337689,51.337689,0,0


In [34]:
df["VendorID"].value_counts(normalize=True)

VendorID
2    0.55066
1    0.44934
Name: proportion, dtype: float64

In [30]:
# df.to_csv("./data/df_output.csv", index=False)